# 토스 경진대회: 클릭 예측 (PBL-3 + PBL-4 기반)

**홍익대학교 3학년 | AI 경진대회 준비**

- **PBL-3**: 전처리 (결측, 스케일링, 불균형, 이상치)
- **PBL-4**: 모델 설계 (Baseline → Interaction → Polynomial → kNN)
- **R 실습 연계**: `Smarket`처럼 시간 분리 + LDA/QDA/kNN 비교
- **부스팅 + 전통 모델 비교**

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 디렉토리 설정
os.chdir(r'C:\Users\tkdwl\Desktop\토스 경진대회')

# 데이터 로드
df_train = pd.read_parquet('train.parquet')
df_test = pd.read_parquet('test.parquet')

print(f"Train: {df_train.shape}, Test: {df_test.shape}")
print(f"Click rate: {df_train['clicked'].mean():.4f}")

## PBL-3: 데이터 전처리

In [ ]:
target_col = 'clicked'
id_col = 'ID'

# 1. 결측치 처리
high_missing = df_train.isnull().mean()
cols_to_drop = high_missing[high_missing > 0.05].index.tolist()
cols_to_drop = [c for c in cols_to_drop if c not in [target_col, id_col]]

df_train = df_train.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

# 수치형 결측 행 제거 (R 실습처럼 clean data)
num_cols = df_train.select_dtypes(include=np.number).columns
num_cols = [c for c in num_cols if c != target_col]

df_train = df_train.dropna(subset=num_cols)
df_test = df_test.dropna(subset=num_cols)

# 2. 이상치 제거 (IQR 기반, R 실습 참고)
def remove_outliers(df, cols, factor=1.5):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        df = df[(df[col] >= lower) & (df[col] <= upper)]
    return df

df_train = remove_outliers(df_train, ['Volume'] if 'Volume' in df_train.columns else [], 3.0)

# 3. hour 문자열 → 정수
df_train['hour'] = pd.to_numeric(df_train['hour'], errors='coerce')
df_test['hour'] = pd.to_numeric(df_test['hour'], errors='coerce')
median_hour = df_train['hour'].median()
df_train['hour'] = df_train['hour'].fillna(median_hour)
df_test['hour'] = df_test['hour'].fillna(median_hour)

# 4. 스케일링 (PBL-3 필수)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_cols = [c for c in num_cols if c not in ['clicked', 'cluster']]
df_train[scaled_cols] = scaler.fit_transform(df_train[scaled_cols])
df_test[scaled_cols] = scaler.transform(df_test[scaled_cols])

print(f"전처리 후: Train {df_train.shape}, Test {df_test.shape}")

## PBL-4: 모델 설계 (R 실습 기반)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
import lightgbm as lgb

# 시간 기반 분리 (R 실습: Year < 2005)
# 여기선 seq 길이로 proxy 사용 (최신 데이터는 길이가 길 수 있음)
df_train['seq_len'] = df_train['seq'].str.split(',').str.len()
threshold = df_train['seq_len'].quantile(0.8)
train_mask = df_train['seq_len'] < threshold

X_train = df_train[train_mask].drop(columns=[target_col, id_col, 'seq', 'seq_len'])
X_valid = df_train[~train_mask].drop(columns=[target_col, id_col, 'seq', 'seq_len'])
y_train = df_train[train_mask][target_col]
y_valid = df_train[~train_mask][target_col]

X_test = df_test.drop(columns=[id_col, 'seq'])

print(f"시간 분리: Train {X_train.shape}, Valid {X_valid.shape}")

### 1. Baseline: Logistic Regression

In [ ]:
logit = LogisticRegression(max_iter=1000)
logit.fit(X_train[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_train.iloc[:, :2], y_train)
pred = logit.predict_proba(X_valid[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_valid.iloc[:, :2])[:, 1]
auc1 = roc_auc_score(y_valid, pred)
print(f"[1] Logistic (Baseline) AUC: {auc1:.4f}")

### 2. + Interaction Term

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_int = poly.fit_transform(X_train[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_train.iloc[:, :2])
X_valid_int = poly.transform(X_valid[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_valid.iloc[:, :2])

logit_int = LogisticRegression(max_iter=1000)
logit_int.fit(X_train_int, y_train)
pred_int = logit_int.predict_proba(X_valid_int)[:, 1]
auc2 = roc_auc_score(y_valid, pred_int)
print(f"[2] +Interaction AUC: {auc2:.4f}")

### 3. + Polynomial (degree=3)

In [ ]:
poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly3.fit_transform(X_train[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_train.iloc[:, :2])
X_valid_poly = poly3.transform(X_valid[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_valid.iloc[:, :2])

logit_poly = LogisticRegression(max_iter=2000)
logit_poly.fit(X_train_poly, y_train)
pred_poly = logit_poly.predict_proba(X_valid_poly)[:, 1]
auc3 = roc_auc_score(y_valid, pred_poly)
print(f"[3] +Polynomial(3) AUC: {auc3:.4f}")

### 4. kNN (k=1,3,5)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_results = {}
for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_train.iloc[:, :2], y_train)
    pred_knn = knn.predict_proba(X_valid[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_valid.iloc[:, :2])[:, 1]
    auc = roc_auc_score(y_valid, pred_knn)
    knn_results[k] = auc
    print(f"[4] kNN(k={k}) AUC: {auc:.4f}")

best_k = max(knn_results, key=knn_results.get)

### 5. LDA / QDA (R 실습)

In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_train.iloc[:, :2], y_train)
pred_lda = lda.predict_proba(X_valid[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_valid.iloc[:, :2])[:, 1]
auc_lda = roc_auc_score(y_valid, pred_lda)
print(f"LDA AUC: {auc_lda:.4f}")

qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_train.iloc[:, :2], y_train)
pred_qda = qda.predict_proba(X_valid[['Lag1', 'Lag2']] if 'Lag1' in X_train.columns else X_valid.iloc[:, :2])[:, 1]
auc_qda = roc_auc_score(y_valid, pred_qda)
print(f"QDA AUC: {auc_qda:.4f}")

### 6. 부스팅 모델 (최종 제출용)

In [ ]:
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'verbose': -1
}

model = lgb.train(params, lgb_train, num_boost_round=2000,
                  valid_sets=[lgb_valid], early_stopping_rounds=100, verbose_eval=100)

final_auc = model.best_score['valid_0']['auc']
print(f"\n최종 LightGBM AUC: {final_auc:.5f}")

## 제출 파일 생성

In [ ]:
pred_test = model.predict(X_test)
submission = pd.DataFrame({'ID': df_test[id_col], 'clicked': pred_test})
submission.to_csv('submission.csv', index=False)
print("제출 완료: submission.csv")

## 성능 요약 (PBL-4 프레젠테이션용)

In [ ]:
results = {
    'Logistic (Baseline)': auc1,
    '+ Interaction': auc2,
    '+ Polynomial(3)': auc3,
    f'kNN(k={best_k})': knn_results[best_k],
    'LDA': auc_lda,
    'QDA': auc_qda,
    'LightGBM (Final)': final_auc
}

pd.Series(results).sort_values(ascending=False)